# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevinwdt/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis and time window

### Data contract

1. **Unit of analysis:** In the final feature frame, one row represents one pseudonymized webpage for one pseudonymized client.

2. **Source table:** I will use `fact_content_daily_performance`. The source table has one row per report date, client, and content item. I will aggregate those daily rows into one row per content item.

3. **Time window:** I will use the March 2026 partition. Features will use March 1–21, 2026. The provisional outcome will use March 22–31, 2026. This separates information known at the decision moment from the later outcome.

4. **Output or proxy:** My Lane 4 project will rank pages by their probability of having lower future CTR than pages in a similar average-position range.

5. **Deliberate exclusion:** I will exclude outcome CTR and the outcome CTR gap from the model features because they are calculated from the period I am trying to predict. Using them would reveal the answer.

My output will support an SEO specialist or content editor deciding which pages deserve CTR, metadata, intent, content, or engagement review first.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install -U duckdb huggingface_hub scikit-learn

import os
import duckdb
import pandas as pd
import numpy as np

# Load the token without printing or storing it in the notebook.
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Add it through the Colab Secrets panel."
    )

# Connect DuckDB to the gated Hugging Face dataset.
con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# Use a mid-panel month, not the final June sample.
MARCH = (
    "read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    ")"
)

print("DuckDB connection created.")
print("Working partition: March 2026")
print("The token was loaded without being displayed.")

DuckDB connection created.
Working partition: March 2026
The token was loaded without being displayed.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features — maximum five

1. **past_impressions:** Total impressions from March 1–21. It is knowable at the decision moment because this search exposure already occurred.

2. **past_ctr_pct:** CTR calculated from clicks and impressions from March 1–21. It is knowable because both values come from the completed feature window.

3. **past_avg_position:** Impression-weighted average search position from March 1–21. It is knowable because it uses only earlier search measurements.

4. **past_position_std:** Variation in average position during March 1–21. It is knowable because it uses only position values observed before the outcome window.

5. **past_active_days:** Number of feature-window days on which the page received impressions. It is knowable because those days have already occurred.

### Label or proxy

`low_ctr_label` is my provisional binary proxy. It equals 1 when the page's outcome-window CTR is below the typical outcome CTR for pages in a similar prior average-position band.

This is a proxy for review priority, not proof that the page has a bad title or should automatically be changed.

### Context

- `client_hash_id`: used for grouped train/test splitting, never as a model feature.
- `content_hash_id`: used to maintain the unit of analysis, never as a model feature.
- `position_band`: used to compare pages with similar search positions.
- `report_date`: used to create the feature and outcome windows.

### Excluded

- `outcome_ctr_pct`: future information used to construct the proxy.
- `outcome_ctr_gap`: directly determines the label and would cause leakage.
- Raw IDs are excluded from model training because they are identifiers rather than meaningful predictive measurements.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

field_contract = pd.DataFrame(
    [
        ["past_impressions", "Feature", "Known from March 1-21"],
        ["past_ctr_pct", "Feature", "Known from March 1-21"],
        ["past_avg_position", "Feature", "Known from March 1-21"],
        ["past_position_std", "Feature", "Known from March 1-21"],
        ["past_active_days", "Feature", "Known from March 1-21"],
        ["low_ctr_label", "Label / proxy", "Defined from March 22-31"],
        ["client_hash_id", "Context", "Used only for grouped splitting"],
        ["content_hash_id", "Context", "Used only to preserve row identity"],
        ["outcome_ctr_pct", "Excluded feature", "Future outcome information"],
        ["outcome_ctr_gap", "Excluded feature", "Directly reveals the label"],
    ],
    columns=["Field", "Role", "Reason"],
)

display(field_contract)

,Field,Role,Reason
0,past_impressions,Feature,Known from March 1-21
1,past_ctr_pct,Feature,Known from March 1-21
2,past_avg_position,Feature,Known from March 1-21
3,past_position_std,Feature,Known from March 1-21
4,past_active_days,Feature,Known from March 1-21
5,low_ctr_label,Label / proxy,Defined from March 22-31
6,client_hash_id,Context,Used only for grouped splitting
7,content_hash_id,Context,Used only to preserve row identity
8,outcome_ctr_pct,Excluded feature,Future outcome information
9,outcome_ctr_gap,Excluded feature,Directly reveals the label


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# VERIFICATION QUERY 1: Grain check
# Expected result: zero duplicate groups.

grain_check = con.sql(
    f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_at_grain
    FROM {MARCH}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
    """
).df()

display(grain_check)

if grain_check.empty:
    print(
        "Grain check passed: no duplicate "
        "date-client-content groups were found."
    )
else:
    print("Warning: the expected source grain did not hold.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,rows_at_grain


Grain check passed: no duplicate date-client-content groups were found.


In [18]:
# VERIFICATION QUERY 2: Lane slice count and date span
# Lane 4 requires impressions and a valid search position.

slice_summary = con.sql(
    f"""
    SELECT
        COUNT(*) AS lane_daily_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(
            DISTINCT (client_hash_id, content_hash_id)
        ) AS content_items,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MARCH}
    WHERE gsc_impressions > 0
      AND gsc_avg_position > 0
    """
).df()

display(slice_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,lane_daily_rows,clients,content_items,first_date,last_date
0,3447872,47,175304,2026-03-01,2026-03-31


In [19]:
# VERIFICATION QUERY 3: GA4 availability
# The assignment specifically requires an IS TRUE filter.

availability_check = con.sql(
    f"""
    WITH lane_rows AS
    (
        SELECT *
        FROM {MARCH}
        WHERE gsc_impressions > 0
          AND gsc_avg_position > 0
    ),
    counts AS
    (
        SELECT
            (SELECT COUNT(*) FROM lane_rows)
                AS rows_before_filter,

            (
                SELECT COUNT(*)
                FROM lane_rows
                WHERE ga4_data_available IS TRUE
            ) AS rows_after_true_filter
    )
    SELECT
        rows_before_filter,
        rows_after_true_filter,
        ROUND(
            100.0 * rows_after_true_filter
            / NULLIF(rows_before_filter, 0),
            2
        ) AS percent_surviving
    FROM counts
    """
).df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_before_filter,rows_after_true_filter,percent_surviving
0,3447872,361041,10.47


### Five-feature frame

The decision moment is the end of March 21, 2026. Every model feature below uses only data available by that date. The later March 22–31 period is used only to construct the provisional outcome.

I require at least 100 prior impressions and 50 outcome impressions to reduce extreme low-volume noise. These are provisional analysis thresholds, not universal business rules.

In [20]:
feature_frame = con.sql(
    f"""
    WITH page_month AS
    (
        SELECT
            client_hash_id,
            content_hash_id,

            -- Feature window: March 1-21
            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN
                    DATE '2026-03-01' AND DATE '2026-03-21'
            ) AS past_impressions,

            SUM(gsc_clicks) FILTER (
                WHERE report_date BETWEEN
                    DATE '2026-03-01' AND DATE '2026-03-21'
            ) AS past_clicks,

            100.0
            * SUM(gsc_clicks) FILTER (
                WHERE report_date BETWEEN
                    DATE '2026-03-01' AND DATE '2026-03-21'
            )
            / NULLIF(
                SUM(gsc_impressions) FILTER (
                    WHERE report_date BETWEEN
                        DATE '2026-03-01' AND DATE '2026-03-21'
                ),
                0
            ) AS past_ctr_pct,

            SUM(gsc_avg_position * gsc_impressions) FILTER (
                WHERE report_date BETWEEN
                    DATE '2026-03-01' AND DATE '2026-03-21'
            )
            / NULLIF(
                SUM(gsc_impressions) FILTER (
                    WHERE report_date BETWEEN
                        DATE '2026-03-01' AND DATE '2026-03-21'
                ),
                0
            ) AS past_avg_position,

            STDDEV_SAMP(gsc_avg_position) FILTER (
                WHERE report_date BETWEEN
                    DATE '2026-03-01' AND DATE '2026-03-21'
                  AND gsc_impressions > 0
            ) AS past_position_std,

            COUNT(DISTINCT report_date) FILTER (
                WHERE report_date BETWEEN
                    DATE '2026-03-01' AND DATE '2026-03-21'
                  AND gsc_impressions > 0
            ) AS past_active_days,

            -- Outcome window: March 22-31
            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN
                    DATE '2026-03-22' AND DATE '2026-03-31'
            ) AS outcome_impressions,

            SUM(gsc_clicks) FILTER (
                WHERE report_date BETWEEN
                    DATE '2026-03-22' AND DATE '2026-03-31'
            ) AS outcome_clicks,

            100.0
            * SUM(gsc_clicks) FILTER (
                WHERE report_date BETWEEN
                    DATE '2026-03-22' AND DATE '2026-03-31'
            )
            / NULLIF(
                SUM(gsc_impressions) FILTER (
                    WHERE report_date BETWEEN
                        DATE '2026-03-22' AND DATE '2026-03-31'
                ),
                0
            ) AS outcome_ctr_pct

        FROM {MARCH}

        WHERE ga4_data_available IS TRUE
          AND gsc_avg_position > 0

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT *
    FROM page_month
    WHERE past_impressions >= 100
      AND outcome_impressions >= 50
      AND past_avg_position BETWEEN 1 AND 20
    """
).df()

print("Feature-frame rows:", f"{len(feature_frame):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 11,915


In [21]:
# Compare pages only with pages in a similar prior position range.
feature_frame["position_band"] = pd.cut(
    feature_frame["past_avg_position"],
    bins=[0, 3, 10, 20],
    labels=["1-3", "4-10", "11-20"],
    include_lowest=True,
)

feature_frame["expected_outcome_ctr_pct"] = (
    feature_frame.groupby(
        "position_band",
        observed=True
    )["outcome_ctr_pct"].transform("median")
)

# Positive gap = outcome CTR fell below the position-band median.
feature_frame["outcome_ctr_gap"] = (
    feature_frame["expected_outcome_ctr_pct"]
    - feature_frame["outcome_ctr_pct"]
)

feature_frame["low_ctr_label"] = (
    feature_frame["outcome_ctr_gap"] > 0
).astype(int)

feature_columns = [
    "past_impressions",
    "past_ctr_pct",
    "past_avg_position",
    "past_position_std",
    "past_active_days",
]

print("One dataframe row = one pseudonymized client-content item.")
print("Five model features:", feature_columns)
print(
    "Positive proxy rate:",
    f"{feature_frame['low_ctr_label'].mean():.1%}"
)

# Do not display IDs or raw URLs in the public notebook.
display(
    feature_frame[
        feature_columns
        + ["position_band", "low_ctr_label"]
    ].head(10)
)

One dataframe row = one pseudonymized client-content item.
Five model features: ['past_impressions', 'past_ctr_pct', 'past_avg_position', 'past_position_std', 'past_active_days']
Positive proxy rate: 49.9%


,past_impressions,past_ctr_pct,past_avg_position,past_position_std,past_active_days,position_band,low_ctr_label
0,444.0,0.675676,8.211712,2.588007,5,4-10,1
1,117.0,0.854701,12.743590,3.089681,6,11-20,1
2,263.0,1.520913,7.581749,0.502679,5,4-10,0
3,145.0,0.000000,11.827586,5.417754,16,11-20,1
4,228.0,3.070175,6.951754,2.050854,10,4-10,0
5,1241.0,0.161160,9.383562,2.455812,13,4-10,1
6,497.0,0.603622,8.152918,1.174643,5,4-10,1
7,2478.0,0.564972,7.356336,0.867320,12,4-10,0
8,131.0,1.526718,7.992366,0.956101,5,4-10,0
9,294.0,1.020408,10.870748,1.533276,7,11-20,0


### Deliberate leakage experiment

I will first train an honest model using only the five feature-window columns.

I will then deliberately add `outcome_ctr_gap`. This is leakage because the label is directly calculated from whether this gap is above zero. The leaky model should therefore appear unrealistically strong.

After demonstrating the problem, I will remove the leaking column and retain the honest score.

In [22]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

model_data = feature_frame.dropna(
    subset=feature_columns
    + [
        "low_ctr_label",
        "outcome_ctr_gap",
        "client_hash_id",
    ]
).copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_index, test_index = next(
    splitter.split(
        model_data,
        model_data["low_ctr_label"],
        groups=model_data["client_hash_id"],
    )
)

train = model_data.iloc[train_index]
test = model_data.iloc[test_index]

# Honest model
honest_model = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=42,
)

honest_model.fit(
    train[feature_columns],
    train["low_ctr_label"],
)

honest_probability = honest_model.predict_proba(
    test[feature_columns]
)[:, 1]

honest_auc = roc_auc_score(
    test["low_ctr_label"],
    honest_probability,
)

# Deliberately leaky model
leaky_columns = feature_columns + ["outcome_ctr_gap"]

leaky_model = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=42,
)

leaky_model.fit(
    train[leaky_columns],
    train["low_ctr_label"],
)

leaky_probability = leaky_model.predict_proba(
    test[leaky_columns]
)[:, 1]

leaky_auc = roc_auc_score(
    test["low_ctr_label"],
    leaky_probability,
)

print(f"Honest ROC-AUC: {honest_auc:.3f}")
print(f"Leaky ROC-AUC:  {leaky_auc:.3f}")
print()
print(
    "The leaky result is not a real improvement. "
    "outcome_ctr_gap directly reveals the label."
)

# Remove the leaking column from the modeling data.
honest_model_frame = model_data.drop(
    columns=["outcome_ctr_gap"]
)

print()
print("outcome_ctr_gap removed.")
print(f"Final score retained for this exercise: {honest_auc:.3f}")

Honest ROC-AUC: 0.775
Leaky ROC-AUC:  1.000

The leaky result is not a real improvement. outcome_ctr_gap directly reveals the label.

outcome_ctr_gap removed.
Final score retained for this exercise: 0.775


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

### Named limitation: availability selection bias

Not every client has the same amount of GSC or GA4 history. By requiring `ga4_data_available IS TRUE`, my feature frame excludes rows and clients without usable analytics coverage. The remaining slice may therefore represent clients with more complete tracking better than clients with short or incomplete histories.

The daily performance table also cannot tell me why a page had low CTR. Low measured CTR could relate to search position, query intent, search-result features, brand familiarity, seasonality, or other factors that are not fully represented here.

Therefore, my proxy can support review prioritization, but it cannot prove that a title or meta-description change will increase clicks.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

rows_before = int(
    availability_check.loc[0, "rows_before_filter"]
)

rows_after = int(
    availability_check.loc[0, "rows_after_true_filter"]
)

rows_removed = rows_before - rows_after
percent_removed = (
    100.0 * rows_removed / rows_before
    if rows_before
    else 0
)

print("Rows removed by the availability requirement:", f"{rows_removed:,}")
print("Percentage removed:", f"{percent_removed:.2f}%")
print(
    "This missingness is a data-coverage limitation, "
    "not evidence of zero engagement."
)

Rows removed by the availability requirement: 3,086,831
Percentage removed: 89.53%
This missingness is a data-coverage limitation, not evidence of zero engagement.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.